# 03. Monolingual, Bilingual & Lexical Resource Statistics
How much training data does each experiment (E1-E7) actually have available, per translation direction?

In [1]:
# ============================================================
# PATH BOOSTER -- guarantees project root in sys.path & CWD.
# Matches notebooks/00_setup_environment.ipynb's convention: this repo
# is deployed both to Colab (fresh `git clone`, folder "Ekegusii-LLM-Translation")
# and to Kineses Cloud / similar Jupyter hosts (pre-placed at
# ~/Ekegusii-LLM-Translation-main -- the "-main" suffix comes from GitHub's
# "Download ZIP" naming). Do not assume either folder name is the cwd.
# ============================================================
import os
import sys

REPO_NAME = "Ekegusii-LLM-Translation"


def _find_project_root():
    if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
        if not os.path.exists(REPO_NAME):
            os.system(f"git clone https://github.com/aykahsay/{REPO_NAME}.git")
            os.system(f"pip install -q -r {REPO_NAME}/requirements.txt")
        return os.path.abspath(REPO_NAME)

    try:
        cwd = os.getcwd()
    except FileNotFoundError:
        cwd = os.path.expanduser("~")
        os.chdir(cwd)

    home = os.path.expanduser("~")
    for candidate in (f"{REPO_NAME}-main", REPO_NAME):
        proj_dir = os.path.join(home, candidate)
        if os.path.isdir(proj_dir):
            return proj_dir

    if os.path.exists("src") and os.path.exists("data"):
        return cwd
    if os.path.basename(cwd) == "notebooks" and os.path.exists(os.path.join("..", "src")):
        return os.path.abspath("..")

    return cwd


project_root = _find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Project root: {project_root}")


Project root: C:\Users\Admin\OneDrive - United States International University (USIU)\Documents\NLP\Multilogual_transaltion_nlp


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
from src.master_corpus.manager import MasterCorpusManager
from src.task_generation.translation_pairs import direction_counts

manager = MasterCorpusManager()
train_df = manager.load_train_split()
counts = direction_counts(train_df)
for direction, count in counts:
    print(f'{direction:25s} {count:>8,} pairs')

INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


INFO | Extracted 37,721 'English'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 37,721 'Ekegusii'->'English' pairs from 39,421 rows.


INFO | Extracted 27,092 'Kiswahili'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 27,092 'Ekegusii'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'English'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'Kiswahili'->'English' pairs from 39,421 rows.


English->Ekegusii           37,721 pairs
Ekegusii->English           37,721 pairs
Kiswahili->Ekegusii         27,092 pairs
Ekegusii->Kiswahili         27,092 pairs
English->Kiswahili          28,792 pairs
Kiswahili->English          28,792 pairs


## Complete triplets available for E4 (Trilingual)

In [4]:
complete_triplets = train_df.dropna(subset=['English', 'Kiswahili', 'Ekegusii'])
print(f'Complete triplets: {len(complete_triplets):,} / {len(train_df):,} '
      f'({100 * len(complete_triplets) / len(train_df):.1f}%)')

Complete triplets: 27,092 / 39,421 (68.7%)


## Lexical corpus resource size (E6)

In [5]:
lexical_df = manager.load_lexical_corpus()
print(f'Lexical entries: {len(lexical_df):,}')
for lang in ['English', 'Kiswahili', 'Ekegusii']:
    non_null = lexical_df[lang].notna().sum()
    print(f'  {lang}: {non_null}/{len(lexical_df)} non-null ({100*non_null/len(lexical_df):.1f}%)')

INFO | Loaded Master Lexical Corpus: 268 terms.


Lexical entries: 268
  English: 0/268 non-null (0.0%)
  Kiswahili: 268/268 non-null (100.0%)
  Ekegusii: 268/268 non-null (100.0%)


## Resource size by experiment (approximate task counts)

In [6]:
from src.experiments.bilingual import BilingualExperiment

e1_pairs = direction_counts(train_df)
eng_eke = sum(c for d, c in e1_pairs if d in ('English->Ekegusii', 'Ekegusii->English'))
swa_eke = sum(c for d, c in e1_pairs if d in ('Kiswahili->Ekegusii', 'Ekegusii->Kiswahili'))
print(f'E1 (English-Ekegusii) task pool: {eng_eke:,}')
print(f'E2 (Swahili-Ekegusii) task pool:  {swa_eke:,}')
print(f'E3 (Combined bilingual) task pool: {eng_eke + swa_eke:,}')

C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO | Extracted 37,721 'English'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 37,721 'Ekegusii'->'English' pairs from 39,421 rows.


INFO | Extracted 27,092 'Kiswahili'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 27,092 'Ekegusii'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'English'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'Kiswahili'->'English' pairs from 39,421 rows.


E1 (English-Ekegusii) task pool: 75,442
E2 (Swahili-Ekegusii) task pool:  54,184
E3 (Combined bilingual) task pool: 129,626
